# Transaction cleaning and population reconciliation

## Summary

The reviewed purchase population contains **527,761 merchandise line items**, including **396,340 identified-customer lines** and **131,421 anonymous lines**. All **541,909 source rows** remain in the audit ledger. Exact copies remain in the baseline; excluding them reduces gross merchandise purchase value by approximately **0.236%**.

These populations represent **gross purchases before credits**, not net revenue. Business SQL analysis and customer segmentation are subsequent stages.

## Context and methods

Source: Chen, D. (2015), [UCI Online Retail](https://doi.org/10.24432/C5BW33), CC BY 4.0. Source window: 1 December 2010 to 9 December 2011; December 2011 is partial. Grain: one original invoice line, traced by Excel row number. Currency: GBP; source timezone unspecified.

### Key assumptions

- Purchase eligibility requires positive quantity and price, reviewed merchandise classification, and no C-prefixed invoice ID.
- Anonymous eligible purchases remain in sales; customer analysis requires an ID.
- Repeated identical lines may be legitimate, so retain them and report a sensitivity.
- Fees, postage, vouchers, samples, and unresolved manual entries are outside merchandise purchases. Classification is inferred from source descriptions.
- Preserve all original strings; normalize only the derived product-code key. Keep extreme values; no credit matching or outlier trimming is assumed.
- Monetary calculations use integer thousandths of GBP, preserving observed prices of GBP 0.001.

See [cleaning methodology](../docs/data_cleaning.md) and `scripts/clean_transactions.py` for complete rules and verification. The following execution regenerates all three local exports from read-only raw SQLite records.

## Data

### Verify provenance and execute cleaning

The pipeline validates the source checksum and full raw-record digest, reconciles status counts and values independently in SQL, and reads every exported field back for comparison before publishing the outputs.

In [1]:
from pathlib import Path
from decimal import Decimal
from hashlib import sha256
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
import clean_transactions as cleaning

summary = cleaning.main()
print(f"Verified original rows: {summary['source_rows']:,}")
print(f"Source SHA-256: {summary['source_sha256']}")
print("Raw columns preserved; three local exports written and read back exactly.")

Verified original rows: 541,909
Source SHA-256: 43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d
Raw columns preserved; three local exports written and read back exactly.


## Results

### Exclusive row-status bridge

Statuses use precedence to avoid double counting: cancellation, nonpositive quantity, nonpositive price, non-merchandise, unresolved item, eligible purchase. Original fields and additional flags preserve overlapping conditions.

In [2]:
status = pd.DataFrame(summary["status_counts"].items(), columns=["Status", "Source rows"])
status["Share of source (%)"] = status["Source rows"] / summary["source_rows"] * 100
assert status["Source rows"].sum() == summary["source_rows"]
display(status.style.format({"Source rows": "{:,.0f}", "Share of source (%)": "{:.2f}"}).hide(axis="index"))

Status,Source rows,Share of source (%)
eligible_purchase,"527,761",97.39
cancellation,"9,288",1.71
non_merchandise,"2,021",0.37
nonpositive_quantity,"1,336",0.25
nonpositive_price,"1,181",0.22
unresolved_item,322,0.06


### Purchase populations and customer coverage

The customer and anonymous populations partition sales rows and purchase value. Distinct entity counts are not generally additive. All-recorded-line value includes signed adjustments and must not be compared as though it were the same population as gross purchases.

In [3]:
def gbp(value):
    return f"{Decimal(value) / Decimal(1000):,.3f}"

populations = pd.DataFrame.from_dict(summary["populations"], orient="index")
populations["Line value (GBP)"] = populations["line_value_millipounds"].map(gbp)
populations = populations.drop(columns="line_value_millipounds").rename(columns={
    "rows": "Rows", "invoice_ids": "Invoice IDs", "known_customer_ids": "Known customer IDs", "quantity": "Quantity"
})
display(populations.style.format({column: "{:,.0f}" for column in ["Rows", "Invoice IDs", "Known customer IDs", "Quantity"]}))

,Rows,Invoice IDs,Known customer IDs,Quantity,Line value (GBP)
all_recorded_lines,"541,909","25,900","4,372","5,176,450","9,747,747.934"
merchandise_purchases,"527,761","19,773","4,334","5,577,103","10,271,433.063"
identified_customer_purchases,"396,340","18,402","4,334","5,157,357","8,761,066.653"
anonymous_purchases,"131,421","1,371",0,"419,746","1,510,366.410"


### Sensitivity to exact copies

Exact equality uses all eight original source fields, ignoring the added lineage number. The alternative retains the earliest source row in each equality group. This is a sensitivity scenario; it does not establish the correct business total.

In [4]:
sensitivity = []
for name, values in summary["duplicate_sensitivity"].items():
    baseline = values["baseline"]
    alternative = values["without_exact_copies"]
    reduction = baseline["line_value_millipounds"] - alternative["line_value_millipounds"]
    sensitivity.append({
        "Population": name,
        "Baseline rows": baseline["rows"],
        "Rows without copies": alternative["rows"],
        "Value reduction (GBP)": gbp(reduction),
        "Value reduction (%)": reduction / baseline["line_value_millipounds"] * 100,
    })
display(pd.DataFrame(sensitivity).style.format({
    "Baseline rows": "{:,.0f}", "Rows without copies": "{:,.0f}", "Value reduction (%)": "{:.3f}"
}).hide(axis="index"))

Population,Baseline rows,Rows without copies,Value reduction (GBP),Value reduction (%)
merchandise_purchases,"527,761","522,540","24,213.740",0.236
identified_customer_purchases,"396,340","391,153","23,839.010",0.272


### Item-code review

Conventional numeric merchandise codes can have multiple letter suffixes. The explicit registry covers exceptional formats and exclusions. For example, real merchandise with a DCGS prefix is retained; manual entries remain unresolved. A previously unseen code outside the conventional pattern stops execution for review.

In [5]:
rules = pd.read_csv(ROOT / "docs" / "item_code_classification.csv")
registry_summary = rules.groupby("item_class").size().rename("Reviewed exception codes").to_frame()
display(registry_summary)
print("Complete registry: docs/item_code_classification.csv")
print("Row classifications, including all overlapping statuses:", summary["item_class_counts"])

,Reviewed exception codes
item_class,
merchandise,8
non_merchandise,14
unresolved,10


Complete registry: docs/item_code_classification.csv
Row classifications, including all overlapping statuses: {'merchandise': 538954, 'non_merchandise': 2374, 'unresolved': 581}


### Validation and output integrity

Export hashes identify the exact local file set. Source rows, quantities, and values are retained at invoice-line grain; no invoice-dimension constraint has been imposed.

In [6]:
for name, expected_hash in summary["export_sha256"].items():
    assert sha256((ROOT / "data" / "processed" / name).read_bytes()).hexdigest() == expected_hash
assert all(summary["checks"].values())
display(pd.DataFrame(summary["checks"].items(), columns=["Check", "Passed"]).style.hide(axis="index"))
print("Export hashes verified for all three files.")

Check,Passed
raw_columns_unchanged,True
raw_digest_verified,True
unique_source_lineage,True
status_partition,True
independent_sql_status_rows_and_value,True
customer_partition_rows_and_value,True
exports_round_trip,True


Export hashes verified for all three files.


## Takeaways

- Cleaning preserves the full source audit trail while separating sales eligibility from customer eligibility.
- Anonymous purchases are included in merchandise totals, so customer-level outputs have narrower coverage.
- Exact-copy removal changes gross purchase value by about 0.236%; the retained-row baseline and sensitivity are both disclosed.
- Gross purchases include transactions that may later have been credited. No claim of net retained spending, profit, or refund-adjusted customer value is supported.
- The next stage will load the reviewed populations into SQLite and reconcile saved business queries. RFM scoring and the partial-month comparison method remain for their respective stages.